# TS-FGES Quickstart (Tetrad + Tigramite format)

This notebook is a minimal, reproducible test.
It installs required Python packages, generates synthetic data, runs TS-FGES, and returns a Tigramite-style graph.

In [1]:
%pip install -q jpype1 numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys, os
import numpy as np

# notebook lives at src/tsfges/test_tsfges/, so go 3 levels up to reach project root
root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
sys.path.insert(0, os.path.join(root, 'src'))
sys.path.insert(0, os.path.join(root, 'utils'))

from time_series_gen import generate_random_contemp_model, generate_nonlinear_contemp_timeseries
from tsfges import run_tsfges

In [5]:
def lin_f(x):
    return x

seed = 42
rs = np.random.RandomState(seed)
N = 4
L = 8
lag_max = 2

links_coeffs = generate_random_contemp_model(
    N=N,
    L=L,
    coupling_coeffs=np.linspace(-0.4, 0.4, 9).tolist(),
    coupling_funcs=[lin_f],
    auto_coeffs=[0.3],
    tau_max=lag_max,
    contemp_fraction=0.3,
    random_state=rs,
)

T = 600
data, nonstationary = generate_nonlinear_contemp_timeseries(
    links_coeffs,
    T=T,
    random_state=rs,
)

if nonstationary:
    raise RuntimeError('Generated series is nonstationary. Try another seed.')

var_names = [f'X{i}' for i in range(N)]
data.shape

(600, 4)

In [6]:
result = run_tsfges(
    data=data,
    lag_max=lag_max,
    var_names=var_names,
    penalty_discount=2.0,
    faithfulness_assumed=True,
    symmetric_first_step=False,
    replicating=True,
    verbose=False,
    max_degree=-1,
    num_threads=1,
)

graph = result['graph']
val_matrix = result['val_matrix']

print('graph shape:', graph.shape)
print('val_matrix shape:', val_matrix.shape)
print('num non-empty marks:', int(np.sum(graph != '')))

graph shape: (4, 4, 3)
val_matrix shape: (4, 4, 3)
num non-empty marks: 13


In [7]:
# Optional: inspect discovered links by lag
for tau in range(graph.shape[2]):
    idx = np.argwhere(graph[:, :, tau] != '')
    print(f'Lag {tau}: {len(idx)} links')
    for i, j in idx:
        print(f'  {var_names[i]} -(tau={tau})-> {var_names[j]} : {graph[i, j, tau]}')

Lag 0: 4 links
  X1 -(tau=0)-> X3 : -->
  X2 -(tau=0)-> X3 : <--
  X3 -(tau=0)-> X1 : <--
  X3 -(tau=0)-> X2 : -->
Lag 1: 6 links
  X0 -(tau=1)-> X0 : -->
  X0 -(tau=1)-> X3 : -->
  X1 -(tau=1)-> X1 : -->
  X2 -(tau=1)-> X1 : -->
  X2 -(tau=1)-> X2 : -->
  X3 -(tau=1)-> X3 : -->
Lag 2: 3 links
  X1 -(tau=2)-> X0 : -->
  X2 -(tau=2)-> X0 : -->
  X3 -(tau=2)-> X0 : -->
